In [0]:
import os, sqlite3, json, pandas as pd
volume_path = '/Volumes/novamart/bronze/landing'
sim_sql_path = f'{volume_path}/source_simulator/sql_server'
sim_crm_path = f'{volume_path}/source_simulator/crm_api'
sim_kafka_path = f'{volume_path}/source_simulator/kafka_raw'

sqlite_db_path = '/temp/novamart_sql_server.db'

def initialize_database_simulator():
    if os.path.exists(sqlite_db_path):
        os.remove(sqlite_db_path)
    conn = sqlite3.connect(sqlite_db_path)
    pd.read_csv(f"{sim_sql_path}/bronze_sql_server_customers.csv").to_sql('customers', conn, if_exists='replace', index=False)
    pd.read_csv(f"{sim_sql_path}/bronze_sql_server_products.csv").to_sql('products', conn, if_exists='replace', index=False)
    pd.read_csv(f"{sim_sql_path}/bronze_sql_server_sales_transactions.csv").to_sql('sales_transactions', conn, if_exists='replace', index=False)
    conn.close()
    
def initialize_database_crm(extract_date):
    if extract_date == '05-07-2026' :
        file_path = f'{sim_crm_path}/bronze_crm_customers_2026-07-05.json'
    elif extract_date == '12-07-2026' :
        file_path = f'{sim_crm_path}/bronze_crm_customers_2026-07-12.json'
    else :
        return {"error": f"No CRM data available for date: {extract_date}"}
    
    try :
        with open(file_path, 'r') as file:
            return json.load(file)
    except FileNotFoundError:
        return {"error": f"File not found: {file_path}"}
    
    print("CRM API Simulator endpoint function is compiled.")

    kafka_live_dir = f"{volume_path}/source_simulator/kafka_live_stream"
    dbutils.fs.mkdirs(kafka_live_dir)

    if os.path.exists(f'{sim_kafka_path}/bronze_kafka_website_clickstream.jsonl'):
         print("✓ Kafka clickstream raw data verified.")
    else:
        print("⚠ Clickstream events file not found in kafka_raw.")

    initialize_database_simulator()
    

In [0]:
# === SIMULATED KAFKA PRODUCER FUNCTION ===
import time
import json
import os

def run_kafka_producer(num_files=5, delay_seconds=10):
    """
    Simulates Kafka: Reads clickstream raw file, splits it into chunks,
    and drops mini JSON files into the active streaming folder.
    """
    raw_file_path = f"{sim_kafka_path}/clickstream_events.json"
    
    # 1. Check if the original raw file exists
    if not os.path.exists(raw_file_path):
        print(f"Error: clickstream_events.json not found at: {raw_file_path}")
        return
        
    # 2. Read and parse the file (safe check for both JSON array and Line-delimited JSON formats)
    try:
        with open(raw_file_path, "r") as f:
            content = f.read().strip()
            if content.startswith("["):
                events = json.loads(content)
            else:
                events = [json.loads(line) for line in content.split("\n") if line.strip()]
    except Exception as e:
        print(f"Error reading clickstream file: {e}")
        return

    total_events = len(events)
    if total_events == 0:
        print("Raw clickstream file is empty.")
        return
        
    # Calculate batch size based on total event count
    chunk_size = max(1, total_events // num_files)
    
    print(f"Starting simulated Kafka Producer...")
    print(f"Dropping {num_files} files into: '{kafka_live_dir}'")
    
    for i in range(num_files):
        start_idx = i * chunk_size
        end_idx = min(start_idx + chunk_size, total_events)
        chunk = events[start_idx:end_idx]
        
        if not chunk:
            break
            
        # Append current epoch time to ensure a unique file name for each batch
        file_name = f"clickstream_batch_{i+1}_{int(time.time())}.json"
        output_path = f"{kafka_live_dir}/{file_name}"
        
        # Write the mini batch JSON file
        with open(output_path, "w") as out_file:
            json.dump(chunk, out_file)
            
        print(f"[Producer] Dropped file {i+1}/{num_files}: {file_name} ({len(chunk)} events)")
        
        # Wait before dropping the next file to simulate real-time streaming intervals
        if i < num_files - 1:
            time.sleep(delay_seconds)
            
    print("✓ Kafka Producer stream simulation completed successfully.")